In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier


In [2]:
df=pd.read_csv(r'C:\Users\ADMIN\OneDrive\Desktop\5thsem_OJT\project1_diabetes\data\processed\dilna_P1W1_features.csv')

In [3]:
feature_cols=['age','pregnancies','glucose','blood_pressure','skin_thickness','insulin','bmi','diabetes_pedigree','bmi category','glucose category']

x=df[feature_cols].copy()
y=df['outcome']
x

,age,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,bmi category,glucose category
0,79,0,124.0,71.0,30.0,124.0,23.4,0.588,normal,prediabetic
1,37,0,153.0,85.0,24.0,42.0,33.9,0.192,obese,diabetic
2,39,0,142.0,68.0,22.0,159.0,26.9,0.777,over weight,diabetic
3,68,7,121.0,88.0,26.0,124.0,29.0,1.217,over weight,prediabetic
4,75,0,107.0,84.0,31.0,101.0,21.2,0.278,normal,prediabetic
...,...,...,...,...,...,...,...,...,...,...
945,64,2,117.0,81.0,34.0,124.0,40.3,0.641,obese,prediabetic
946,71,0,140.0,64.0,37.0,124.0,33.4,0.116,obese,diabetic
947,40,0,130.0,73.0,26.0,258.0,20.1,0.709,normal,diabetic
948,52,4,123.0,76.0,22.0,189.0,38.9,0.540,obese,prediabetic


In [4]:
x=pd.get_dummies(x,drop_first=True)
x.shape

(950, 13)

In [5]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [6]:
print("train of x =",x_train.shape,"test",x_test.shape,)
print()
print("train of y =",y_train.shape,"test",y_test.shape)

train of x = (760, 13) test (190, 13)

train of y = (760,) test (190,)


In [7]:
print("y train",y_train.value_counts(normalize=True))
print()
print("y test",y_test.value_counts(normalize=True))

y train outcome
0    0.713158
1    0.286842
Name: proportion, dtype: float64

y test outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64


In [8]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)#Calculates the mean and standard deviation from x train
x_test_scaled=scaler.transform(x_test)#Uses those values to standardize x_train.s

In [9]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)

In [10]:
print('LogisticRegression',accuracy_score(y_test,logreg.predict(x_test_scaled)))
print('DecisionTreeClassifier',accuracy_score(y_test,dtree.predict(x_test)))
print('kneighborsclassifier',accuracy_score(y_test,knn.predict(x_test_scaled)))

LogisticRegression 0.7526315789473684
DecisionTreeClassifier 0.7210526315789474
kneighborsclassifier 0.7473684210526316


In [11]:
models = [
    ("Logistic Regression", logreg, x_test_scaled),
    ("Decision Tree (d=3)", dtree, x_test),
    ("KNN (k=25)", knn, x_test_scaled)
]

print("\nTest set:", len(y_test), "patients |",
      int(np.sum(y_test)), "of them diabetic")


Test set: 190 patients | 55 of them diabetic


In [12]:
rows = []

for name, m, Xt in models:
    cm = confusion_matrix(y_test, m.predict(Xt))
    tn, fp, fn, tp = cm.ravel()

    rows.append({
        "model": name,
        "correct_negatives": int(tn),
        "false_alarms": int(fp),
        "patients_missed": int(fn),
        "patients_found": int(tp),
        "accuracy": round(accuracy_score(y_test, m.predict(Xt)), 4)
    })

matrices = pd.DataFrame(rows)
print(matrices.to_string(index=False))

              model  correct_negatives  false_alarms  patients_missed  patients_found  accuracy
Logistic Regression                127             8               39              16    0.7526
Decision Tree (d=3)                121            14               39              16    0.7211
         KNN (k=25)                132             3               45              10    0.7474


* **127 (TN):** 127 people were correctly identified as not having diabetes.
* **8 (FP):** 8 people were incorrectly identified as having diabetes.
* **39 (FN):** 39 people who actually had diabetes were missed by the model.
* **16 (TP):** 16 people who had diabetes were correctly identified by the model.

* **Logistic Regression:** The main error is **missed patients (FN = 39)**.
* **Decision Tree:** The main error is **missed patients (FN = 39)**, along with 14 false alarms.
* **KNN:** The main error is **missed patients (FN = 45)**, which is the highest among the three models.

# score the model that never says yes

In [13]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_train,y_train)
y_pred = dummy.predict(x_test)
confusion_matrix(y_test,y_pred)

array([[135,   0],
       [ 55,   0]])

In [14]:
print('DummyClassifier',accuracy_score(y_test,y_pred))

DummyClassifier 0.7105263157894737


In [15]:
y_test.value_counts(normalize=True)

outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64

# Recall

Out of all the actual positive patients, how many did the model correctly 

Formula:

recall = True Positive (TP)/(True Positive (TP)+False Negative(FN))

In [16]:
cm =confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print("recall:tp/tp+fn=",(tp/(tp+fn)))

recall:tp/tp+fn= 0.2909090909090909


recall

In [17]:
from sklearn.metrics import recall_score

In [18]:
recall_score(y_test,logreg.predict(x_test_scaled))
print("LogisticRegression Recall =",recall_score(y_test,logreg.predict(x_test_scaled)))
print("DecisionTreeClassifier  Recall =",recall_score(y_test,dtree.predict(x_test)))
print("KNeighborsClassifier  Recall =",recall_score(y_test,knn.predict(x_test_scaled)))


LogisticRegression Recall = 0.2909090909090909
DecisionTreeClassifier  Recall = 0.2909090909090909
KNeighborsClassifier  Recall = 0.18181818181818182


Precision

In [19]:
cm =confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print("precision:tp/tp+fp=",(tp/(tp+fp)))

precision:tp/tp+fp= 0.6666666666666666


# Precision

Out of all the patients the model predicted as positive, how many were actually positive

Precision = True Positives(TP)/(True Positives (TP)+False Positives(FP))

In [20]:
from sklearn.metrics import precision_score

In [21]:
precision_score(y_test,logreg.predict(x_test_scaled))
print("LogisticRegression precision =",precision_score(y_test,logreg.predict(x_test_scaled)))
print("DecisionTreeClassifier  precision =",precision_score(y_test,dtree.predict(x_test)))
print("KNeighborsClassifier  precision =",precision_score(y_test,knn.predict(x_test_scaled)))


LogisticRegression precision = 0.6666666666666666
DecisionTreeClassifier  precision = 0.5333333333333333
KNeighborsClassifier  precision = 0.7692307692307693
